# 06 — Few-shot: Taxonomy Classification\n
\n
Evaluate k-shot fine-tuning on FishNet top-100 species.\n
\n
Planned k values: 1, 5, 10, 20.\n
Report mean ± std over multiple random seeds/splits.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch

ROOT = Path('..').resolve()
sys.path.append(str((ROOT / 'src').resolve()))

MANIFEST_DIR = (ROOT / 'data' / 'manifests').resolve()
OUT_DIR = (ROOT / 'outputs' / 'fewshot_taxonomy').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else 'cpu'
)
print('device:', device)
print('OUT_DIR:', OUT_DIR)


In [ ]:
from torch.utils.data import DataLoader

from fish_ai.data.fewshot import write_fewshot_train_manifest
from fish_ai.data.jsonl import read_jsonl
from fish_ai.data.taxonomy_dataset import FishTaxonomyDataset
from fish_ai.models.taxonomy_classifier import TaxonomyClassifier, TaxonomyHeadSizes
from fish_ai.train.taxonomy_train import TrainConfig, build_label_maps, evaluate, train_one_epoch
from fish_ai.utils.run_logging import write_csv, write_json

train_full = MANIFEST_DIR / 'fishnet_taxonomy_train.jsonl'
val_manifest = MANIFEST_DIR / 'fishnet_taxonomy_val.jsonl'
test_manifest = MANIFEST_DIR / 'fishnet_taxonomy_test.jsonl'

print(train_full.exists(), val_manifest.exists(), test_manifest.exists())


In [ ]:
# Build label maps from FULL training set (keeps label space consistent across k)
train_rows = []
for r in read_jsonl(train_full):
    tax = r['taxonomy']
    train_rows.append({'family': tax['family'], 'genus': tax['genus'], 'species': tax['species']})

maps = build_label_maps(train_rows)

sizes = TaxonomyHeadSizes(
    n_family=len(maps['family']),
    n_genus=len(maps['genus']),
    n_species=len(maps['species']),
)

sizes


In [ ]:
def run_kshot(k: int, seed: int, backbone: str = 'efficientnet_b0', epochs: int = 5):
    fewshot_manifest = OUT_DIR / f'fishnet_taxonomy_train_k{k}_seed{seed}.jsonl'
    write_fewshot_train_manifest(train_full, fewshot_manifest, k=k, seed=seed)

    ds_train = FishTaxonomyDataset(fewshot_manifest, image_size=224, augment=True)
    ds_val = FishTaxonomyDataset(val_manifest, image_size=224, augment=False)
    dl_train = DataLoader(ds_train, batch_size=64, shuffle=True, num_workers=2)
    dl_val = DataLoader(ds_val, batch_size=64, shuffle=False, num_workers=2)

    model = TaxonomyClassifier(sizes, backbone=backbone, pretrained=True).to(device)
    cfg = TrainConfig(num_epochs=epochs, batch_size=64, num_workers=2, lr=3e-4)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    hist = []
    for epoch in range(cfg.num_epochs):
        tr = train_one_epoch(model, dl_train, opt, device=device, maps=maps)
        va = evaluate(model, dl_val, device=device, maps=maps)

        row = {'k': k, 'seed': seed, 'epoch': epoch + 1, **tr}
        for lvl in ['family', 'genus', 'species']:
            for key, val in va[lvl].items():
                row[f'val_{lvl}_{key}'] = val

        hist.append(row)
        print(row)

    run_dir = OUT_DIR / f'k{k}_seed{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'model_state': model.state_dict(),
            'backbone': model.backbone_name,
            'maps': maps,
            'k': k,
            'seed': seed,
        },
        run_dir / 'model.pt',
    )
    write_csv(run_dir / 'history.csv', hist)
    write_json(run_dir / 'val_metrics_last.json', va)
    return va, hist


def summarize(results):
    # results: list of (k, seed, metrics_dict)
    out = []
    for k in sorted({r[0] for r in results}):
        rows = [r for r in results if r[0] == k]
        vals = [rr[2]['species']['acc_top1'] for rr in rows]
        out.append(
            {
                'k': k,
                'n_seeds': len(vals),
                'species_acc_top1_mean': float(np.mean(vals)),
                'species_acc_top1_std': float(np.std(vals)),
            }
        )
    return out


In [ ]:
ks = [1, 5, 10, 20]
seeds = [0, 1, 2]

results = []
for k in ks:
    for seed in seeds:
        va, hist = run_kshot(k=k, seed=seed, backbone='efficientnet_b0', epochs=5)
        results.append((k, seed, va))

summary = summarize(results)
summary


In [ ]:
from fish_ai.utils.run_logging import write_csv

write_csv(OUT_DIR / 'summary.csv', summary)
print('wrote:', OUT_DIR / 'summary.csv')
